In [ ]:
%%configure
{
  "defaultLakehouse": {
    "name": { "parameterName": "LakehouseName", "defaultValue": "AzureCostAnalyzer_LH" },
    "id": { "parameterName": "LakehouseId", "defaultValue": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx" },
    "workspaceId": { "parameterName": "WorkspaceId", "defaultValue": "yyyyyyyy-yyyy-yyyy-yyyy-yyyyyyyyyyyy" }
  }
}

# 06 · Gold · Reservations

Builds **`gold_reservations_coverage`**, **`gold_reservations_waste`** and **`gold_reservations_detail`** — RI/SP coverage, utilization and per-commitment actionable detail, derived entirely from the FOCUS export.

**Outputs**:
- `gold_reservations_coverage`: YearMonth × SubAccountName → ReservedCost, TotalCost, Coverage %
- `gold_reservations_waste`: YearMonth × SubAccountName → Unused (waste) cost, Utilization %
- `gold_reservations_detail`: YearMonth × SubAccountName × Commitment (name/type/category) × Service × Region → UsedCost, UnusedCost, Utilization %, savings — answers **"which reservation/discount is idle so we can act on it"**

**FOCUS semantics used** (per [Microsoft Learn — Validate FOCUS data](https://learn.microsoft.com/cloud-computing/finops/focus/validate)):
- Committed usage → `PricingCategory = 'Committed'` (Reservations + Savings Plans).
- Unused commitment → `ChargeCategory = 'Usage'` AND `CommitmentDiscountStatus = 'Unused'`.
- Used commitment → `CommitmentDiscountStatus = 'Used'`.

**Dependencies**: `focus_partitioned` with `PricingCategory`, `CommitmentDiscountType`, `CommitmentDiscountStatus`, `CommitmentDiscountName`, `CommitmentDiscountCategory`.

> On subscriptions with no reservations/savings plans, all tables return cost = 0 with NULL utilization. That is correct — the metrics populate automatically once you have commitments.



In [ ]:
# Configure default lakehouse using parameters
# mssparkutils is available in Fabric runtime (no import needed)
from pyspark.sql import functions as F
from datetime import date
from dateutil.relativedelta import relativedelta

# Partition pruning: read last 12 months
lookback_date = date.today() - relativedelta(months=12)
year_filter = lookback_date.year
month_filter = lookback_date.month

# ----------------------------------------------------------------------------
# FOCUS commitment-discount semantics (per Microsoft Learn / FOCUS spec):
#   - Committed usage:   PricingCategory = 'Committed'  (covers both
#                        Reservations and Savings Plans; differentiate with
#                        CommitmentDiscountType = 'Reservation' | 'Savings Plan').
#   - Unused commitment: ChargeCategory = 'Usage' AND CommitmentDiscountStatus = 'Unused'
#                        (FOCUS books unused commitment cost in the Usage category).
#   - Used commitment:   CommitmentDiscountStatus = 'Used'.
# These derive real coverage & waste straight from the FOCUS export — no
# external Reservations API needed. On subscriptions without any reservation
# or savings plan, ReservedCost/WasteCost are simply 0 (correct, not a bug).
# ----------------------------------------------------------------------------

# gold_reservations_coverage: committed (RI/SP) cost / total cost by month/sub
coverage_df = spark.sql(f"""
    SELECT
        DATE_FORMAT(ChargePeriodStart, 'yyyy-MM') AS YearMonth,
        SubAccountName,
        SUM(CASE WHEN PricingCategory = 'Committed' THEN EffectiveCost ELSE 0 END) AS ReservedCost,
        SUM(EffectiveCost) AS TotalCost,
        (SUM(CASE WHEN PricingCategory = 'Committed' THEN EffectiveCost ELSE 0 END) /
         NULLIF(SUM(EffectiveCost), 0)) * 100 AS CoveragePercent
    FROM focus_partitioned
    WHERE ChargeCategory = 'Usage'
      AND (Year > {year_filter} OR (Year = {year_filter} AND Month >= {month_filter}))
    GROUP BY YearMonth, SubAccountName
""")

# overwriteSchema: the gold tables may already exist with a different column
# type from a prior run; force the new schema instead of attempting a merge
# (avoids DELTA_FAILED_TO_MERGE_FIELDS).
coverage_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_reservations_coverage")
print(f"✓ Created gold_reservations_coverage with {coverage_df.count():,} rows")

# gold_reservations_waste: unused commitment cost + real utilization %
#   WasteCost          = EffectiveCost where CommitmentDiscountStatus = 'Unused'
#   UtilizationPercent = Used / (Used + Unused) * 100   (NULL when no commitment)
# CAST to DOUBLE so the type is deterministic even when every row is NULL
# (a bare ELSE NULL infers a `void` type and breaks Delta schema merge).
waste_df = spark.sql(f"""
    SELECT
        YearMonth,
        SubAccountName,
        CAST(UnusedCost AS DOUBLE) AS WasteCost,
        CAST(
            CASE
                WHEN (UsedCost + UnusedCost) > 0
                THEN (UsedCost / (UsedCost + UnusedCost)) * 100
                ELSE NULL
            END AS DOUBLE
        ) AS UtilizationPercent
    FROM (
        SELECT
            DATE_FORMAT(ChargePeriodStart, 'yyyy-MM') AS YearMonth,
            SubAccountName,
            SUM(CASE WHEN CommitmentDiscountStatus = 'Used'   THEN EffectiveCost ELSE 0 END) AS UsedCost,
            SUM(CASE WHEN CommitmentDiscountStatus = 'Unused' THEN EffectiveCost ELSE 0 END) AS UnusedCost
        FROM focus_partitioned
        WHERE ChargeCategory = 'Usage'
          AND CommitmentDiscountType IS NOT NULL
          AND (Year > {year_filter} OR (Year = {year_filter} AND Month >= {month_filter}))
        GROUP BY YearMonth, SubAccountName
    )
""")

waste_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_reservations_waste")
print(f"✓ Created gold_reservations_waste with {waste_df.count():,} rows")


In [ ]:
# gold_reservations_detail: per-commitment actionable view.
#   Grain: YearMonth × SubAccountName × Commitment (Id/Name/Type/Category) × Service × Region.
#   Purpose: surface WHICH reservations / savings plans are idle (UnusedCost > 0) so the
#   you can right-size them or deploy matching workloads to consume them.
#   Hybrid savings baseline for committed usage = ContractedCost (on-demand-equivalent at the
#   your negotiated rate); SavingsUsed = ContractedCost - EffectiveCost on Used rows.
detail_df = spark.sql(f"""
    SELECT
        YearMonth,
        SubAccountName,
        CommitmentDiscountId,
        CommitmentDiscountName,
        CommitmentDiscountType,
        CommitmentDiscountCategory,
        ServiceName,
        RegionName,
        UsedCost,
        UnusedCost                                          AS WasteCost,
        (UsedCost + UnusedCost)                             AS TotalCommitmentCost,
        ContractedCostUsed,
        (ContractedCostUsed - UsedCost)                     AS SavingsUsed,
        CASE
            WHEN (UsedCost + UnusedCost) > 0
            THEN (UsedCost / (UsedCost + UnusedCost)) * 100
            ELSE NULL
        END                                                 AS UtilizationPercent
    FROM (
        SELECT
            DATE_FORMAT(ChargePeriodStart, 'yyyy-MM')                                      AS YearMonth,
            SubAccountName,
            CommitmentDiscountId,
            COALESCE(CommitmentDiscountName, '(sin nombre)')                               AS CommitmentDiscountName,
            COALESCE(CommitmentDiscountType, '(desconocido)')                              AS CommitmentDiscountType,
            COALESCE(CommitmentDiscountCategory, '(desconocido)')                          AS CommitmentDiscountCategory,
            ServiceName,
            RegionName,
            SUM(CASE WHEN CommitmentDiscountStatus = 'Used'   THEN EffectiveCost  ELSE 0 END) AS UsedCost,
            SUM(CASE WHEN CommitmentDiscountStatus = 'Unused' THEN EffectiveCost  ELSE 0 END) AS UnusedCost,
            SUM(CASE WHEN CommitmentDiscountStatus = 'Used'   THEN ContractedCost ELSE 0 END) AS ContractedCostUsed
        FROM focus_partitioned
        WHERE ChargeCategory = 'Usage'
          AND PricingCategory = 'Committed'
          AND CommitmentDiscountType IS NOT NULL
          AND (Year > {year_filter} OR (Year = {year_filter} AND Month >= {month_filter}))
        GROUP BY YearMonth, SubAccountName, CommitmentDiscountId,
                 CommitmentDiscountName, CommitmentDiscountType, CommitmentDiscountCategory,
                 ServiceName, RegionName
    )
""")

detail_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_reservations_detail")
print(f"✓ Created gold_reservations_detail with {detail_df.count():,} rows")
detail_df.orderBy(F.desc("WasteCost")).show(20, truncate=False)
